In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the training dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Describe the dataset
print(train_data.describe())

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the distribution of numeric features
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(train_data[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# Visualize the distribution of categorical features
for col in categorical_cols:
    plt.figure(figsize=(8, 6))
    sns.countplot(y=train_data[col])
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

# Correlation matrix for numeric features
plt.figure(figsize=(12, 10))
correlation_matrix = train_data[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numeric Features')
plt.show()


      id  Gender  ...                 MTRANS           NObeyesdad
0   9958    Male  ...             Automobile       Obesity_Type_I
1   7841    Male  ...  Public_Transportation  Insufficient_Weight
2   9293    Male  ...  Public_Transportation      Obesity_Type_II
3  15209  Female  ...             Automobile       Obesity_Type_I
4  16515    Male  ...  Public_Transportation  Overweight_Level_II

[5 rows x 18 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16606 entries, 0 to 16605
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              16606 non-null  int64  
 1   Gender                          16606 non-null  object 
 2   Age                             16606 non-null  float64
 3   Height                          16606 non-null  float64
 4   Weight                          16606 non-null  float64
 5   family_history_with_overweight  16606 no

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 07:26:39.268 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS', 'NObeyesdad'], 'Numeric': ['id', 'Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the datasets
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv')
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/test.csv')

# Separate features and target
train_features = train_data.drop(columns=['id', 'NObeyesdad'])
train_target = train_data['NObeyesdad']
test_features = test_data.drop(columns=['id', 'NObeyesdad'])
test_target = test_data['NObeyesdad']

# Handle missing values
numeric_cols = train_features.select_dtypes(include=[np.number]).columns
categorical_cols = train_features.select_dtypes(include=['object']).columns

# Fill missing values for numeric columns with mean
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
train_features = fill_missing_numeric.fit_transform(train_features)
test_features = fill_missing_numeric.transform(test_features)

# Fill missing values for categorical columns with most frequent
fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
train_features = fill_missing_categorical.fit_transform(train_features)
test_features = fill_missing_categorical.transform(test_features)

# Encode categorical variables
label_encode = LabelEncode(features=categorical_cols)
train_features = label_encode.fit_transform(train_features)
test_features = label_encode.transform(test_features)

# Normalize numerical features
standard_scale = StandardScale(features=numeric_cols)
train_features = standard_scale.fit_transform(train_features)
test_features = standard_scale.transform(test_features)

# Display the preprocessed data
print(train_features.head())
print(test_features.head())


   Gender       Age    Height    Weight  ...       FAF       TUE  CALC  MTRANS
0       1 -1.199372  0.800816  0.343527  ...  2.394982  0.629728     2       0
1       1 -0.213196  0.610243 -1.256602  ...  1.205953  0.629728     2       3
2       1 -0.357492  1.372928  1.294154  ... -0.292365 -1.025610     1       3
3       0  2.976687 -1.379006 -0.302254  ... -1.172105 -1.025610     1       0
4       1 -0.155357  1.144999  0.267553  ...  1.205953  0.629728     0       3

[5 rows x 16 columns]
   Gender       Age    Height    Weight  ...       FAF       TUE  CALC  MTRANS
0       0  0.366650 -0.585344  0.910790  ... -1.094505  0.151883     1       3
1       1 -1.025370  0.571361 -0.302254  ...  0.016924  0.629728     1       3
2       0 -0.092663  0.151229 -0.122871  ...  1.205953 -1.021306     2       3
3       1  1.063013 -0.338806  0.913349  ... -1.172105 -0.836145     1       3
4       1 -1.199372 -1.034823 -1.441869  ... -1.172105  2.285067     2       3

[5 rows x 16 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_features)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Assuming train_features and train_target are already defined from previous steps
# train_features, train_target = ...

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode the target variable
train_target_encoded = label_encoder.fit_transform(train_target)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_features, train_target_encoded, test_size=0.2, random_state=42)

# Initialize the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Predict on the validation set
y_val_pred = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {accuracy:.4f}')

# Prepare the test features
test_features = test_features.drop(columns=['id', 'NObeyesdad'])
test_features = fill_missing_numeric.transform(test_features)
test_features = fill_missing_categorical.transform(test_features)
test_features = label_encode.transform(test_features)
test_features = standard_scale.transform(test_features)

# Predict on the test set
y_test_pred = model.predict(test_features)

# Encode the test target variable
test_target_encoded = label_encoder.transform(test_target)

# Calculate test accuracy
test_accuracy = accuracy_score(test_target_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')


Validation Accuracy: 0.8910


KeyError: "['id', 'NObeyesdad'] not found in axis"